In [14]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42
np.random.seed(SEED)
print('설정 완료')

설정 완료


In [15]:
# ── NSL-KDD 로드 & 전처리 ─────────────────────────────────────
COLS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]
CAT_COLS = ['protocol_type', 'service', 'flag']
BASE = r'c:\Users\kevin\OneDrive\Desktop\AISO\kdd'

def load_kdd(path):
    df = pd.read_csv(path, header=None, names=COLS).drop(columns=['difficulty'])
    for c in CAT_COLS:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    atk = df['label'].copy()
    df['y'] = (df['label'] != 'normal').astype(int)
    X = df.drop(columns=['label','y']).values.astype(float)
    y = df['y'].values
    return X, y, atk

X_train_raw, y_train_raw, atk_train = load_kdd(f'{BASE}/KDDTrain+_20Percent.txt')
X_test_raw,  y_test,      atk_test  = load_kdd(f'{BASE}/KDDTest+.txt')

print(f'Train: {X_train_raw.shape}  anomaly={y_train_raw.mean()*100:.1f}%')
print(f'Test : {X_test_raw.shape}   anomaly={y_test.mean()*100:.1f}%')

train_atk_types = set(atk_train[y_train_raw==1].unique())
test_atk_types  = set(atk_test[y_test==1].unique())
novel = test_atk_types - train_atk_types
print(f'\n학습 공격 유형: {len(train_atk_types)}종')
print(f'테스트 공격 유형: {len(test_atk_types)}종  (미지 공격: {len(novel)}종)')
print(f'미지 공격 예시: {sorted(novel)[:6]}')

Train: (25192, 41)  anomaly=46.6%
Test : (22544, 41)   anomaly=56.9%

학습 공격 유형: 21종
테스트 공격 유형: 37종  (미지 공격: 18종)
미지 공격 예시: ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'perl']


In [16]:
# ── 불균형 시나리오 생성 ─────────────────────────────────────
# 정상: 전체 사용 (13,449)
# 이상: 전체(11,743) 중 10%만 학습에 사용 → 실제 탐지 환경 모사
rng = np.random.RandomState(SEED)

normal_idx   = np.where(y_train_raw == 0)[0]
anom_idx_all = np.where(y_train_raw == 1)[0]

N_NORMAL   = len(normal_idx)                   # 13,449
N_SEEN     = max(int(N_NORMAL * 0.10), 50)     # 1,345 (10%)

seen_anom_idx = rng.choice(anom_idx_all, N_SEEN, replace=False)

X_imbal = np.vstack([X_train_raw[normal_idx], X_train_raw[seen_anom_idx]])
y_imbal = np.array([0]*N_NORMAL + [1]*N_SEEN)
atk_imbal_anom = atk_train.iloc[seen_anom_idx]

# Scale (test도 동일 scaler)
scaler = StandardScaler()
X_imbal = scaler.fit_transform(X_imbal)
X_test  = scaler.transform(X_test_raw)

print(f'불균형 학습셋: {N_NORMAL:,} normal + {N_SEEN:,} anomaly  (이상 비율 {N_SEEN/(N_NORMAL+N_SEEN)*100:.1f}%)')
print(f'테스트셋:      {len(X_test):,} samples  (이상 비율 {y_test.mean()*100:.1f}%)')
print(f'\n학습셋 이상 유형 분포:')
print(atk_imbal_anom.value_counts().to_string())

불균형 학습셋: 13,449 normal + 1,344 anomaly  (이상 비율 9.1%)
테스트셋:      22,544 samples  (이상 비율 56.9%)

학습셋 이상 유형 분포:
label
neptune            967
ipsweep             83
satan               78
portsweep           72
smurf               53
nmap                26
warezclient         24
back                18
teardrop            12
pod                  7
land                 1
multihop             1
buffer_overflow      1
guess_passwd         1


In [ ]:
# ── 평가 함수 + 룰 기반 샘플러 ───────────────────────────────
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

preds = {}   # 분석 셀에서 사용할 예측값 저장

def evaluate(X_tr, y_tr, label=''):
    clf = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    if label:
        preds[label.strip()] = prob
    res  = {
        'PR-AUC': average_precision_score(y_test, prob),
        'F1':     f1_score(y_test, pred, zero_division=0),
        'AUC':    roc_auc_score(y_test, prob),
    }
    if label:
        print(f'  {label:<22} PR-AUC={res["PR-AUC"]:.4f}  F1={res["F1"]:.4f}  AUC={res["AUC"]:.4f}')
    return res

# ── 이상 공간 오버샘플링 공통 헬퍼 ───────────────────────────
N_AG = 30; N_IT = 150; ALPHA = 0.2

def build_train(X_norm, X_anom, anom_idx):
    Xtr = np.vstack([X_norm, X_anom[anom_idx]])
    ytr = np.array([0]*len(X_norm) + [1]*len(anom_idx))
    return Xtr, ytr

def run_random_oversample(X_anom, n_target, seed):
    return np.random.RandomState(seed).choice(len(X_anom), size=n_target, replace=True)

def run_kmeans_oversample(X_anom, n_target, seed, n_clusters=10):
    km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=5).fit(X_anom)
    rng2 = np.random.RandomState(seed)
    per_cluster = n_target // n_clusters
    idx = []
    for c in range(n_clusters):
        pool = np.where(km.labels_ == c)[0]
        if len(pool): idx.extend(rng2.choice(pool, per_cluster, replace=True))
    while len(idx) < n_target:
        idx.append(rng2.choice(len(X_anom)))
    return np.array(idx[:n_target])

def run_greedy_oversample(X_anom, n_target, seed, k=5):
    nn = NearestNeighbors(n_neighbors=min(k+1, len(X_anom))).fit(X_anom)
    dists, _ = nn.kneighbors(X_anom)
    density = 1.0 / (dists[:, 1:].mean(axis=1) + 1e-8)
    probs = 1.0 / (density + 1e-8); probs /= probs.sum()
    return np.random.RandomState(seed).choice(len(X_anom), size=n_target, replace=True, p=probs)

def run_aco_oversample(X_anom, n_target, seed):
    rng2 = np.random.RandomState(seed)
    N_a, D = X_anom.shape
    Xn = (X_anom - X_anom.min(0)) / np.where(X_anom.max(0)-X_anom.min(0)>1e-8,
                                               X_anom.max(0)-X_anom.min(0), 1.0)
    pheromone = np.ones(N_a); visit = np.zeros(N_a)
    for _ in range(N_IT):
        for _ in range(N_AG):
            i = rng2.choice(N_a, p=pheromone/pheromone.sum())
            dists = np.linalg.norm(Xn - Xn[i], axis=1); dists[i] = 1e9
            cands = np.argsort(dists)[:10]
            j = cands[np.argmax(pheromone[cands])]
            nn = np.argmin(np.linalg.norm(Xn - np.clip((Xn[i]+Xn[j])/2, 0, 1), axis=1))
            visit[nn] += 1
        pheromone = pheromone * 0.95 + visit * 0.1
    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, size=n_target, replace=True, p=probs)

def run_pso_oversample(X_anom, n_target, seed):
    rng2 = np.random.RandomState(seed)
    N_a, D = X_anom.shape
    Xn = (X_anom - X_anom.min(0)) / np.where(X_anom.max(0)-X_anom.min(0)>1e-8,
                                               X_anom.max(0)-X_anom.min(0), 1.0)
    X = Xn[rng2.choice(N_a, N_AG, replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = np.array([-np.min(np.linalg.norm(Xn-X[i], axis=1)) for i in range(N_AG)])
    gi = np.argmax(pS); gX = X[gi].copy()
    visit = np.zeros(N_a)
    for _ in range(N_IT):
        r1, r2 = rng2.rand(N_AG, D), rng2.rand(N_AG, D)
        V = 0.729*V + 1.494*r1*(pX-X) + 1.494*r2*(gX-X)
        X = np.clip(X + ALPHA*V, 0, 1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i], axis=1))
            X[i] = Xn[nn]; visit[nn] += 1
            sc = -np.min(np.linalg.norm(Xn-X[i], axis=1))
            if sc > pS[i]: pX[i] = X[i].copy(); pS[i] = sc
        gi = np.argmax(pS); gX = pX[gi].copy()
    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, size=n_target, replace=True, p=probs)

print('평가 함수 + 샘플러 7종 준비 완료')

In [18]:
# ── AISO 오버샘플러 ───────────────────────────────────────────
# type-mediated 척력 → 에이전트가 서로 다른 공격 클러스터로 분산
# neptune(다수) 클러스터에서 밀려나 희귀 공격(satan, smurf 등) 방문 증가
N_TYPES = 12; BETA = 0.08; W_REPEL = 2.0; M_LOW = -0.5

def run_aiso_oversample(X_anom, n_target, seed):
    rng2 = np.random.RandomState(seed)
    N_a, D = X_anom.shape
    Xn = (X_anom - X_anom.min(0)) / np.where(X_anom.max(0)-X_anom.min(0)>1e-8,
                                               X_anom.max(0)-X_anom.min(0), 1.0)
    init_idx = rng2.choice(N_a, N_AG, replace=True)
    X = Xn[init_idx].copy().astype(float)
    W = rng2.dirichlet(np.ones(N_TYPES), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (N_TYPES, N_TYPES))
    visit = np.zeros(N_a)
    w_r = W_REPEL

    for t in range(N_IT):
        if t % 10 == 0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1, N_AG)])
            w_r = 1.0 + 3.0 * np.exp(-div / 0.12)

        C = W @ M @ W.T; np.fill_diagonal(C, 0)

        for i in range(N_AG):
            ci = C[i].copy(); ci[i] = 0
            top_a = np.argsort(ci)[-3:]
            top_r = np.argsort(ci)[:3]
            Fv  = sum(ci[ja] * (X[ja] - X[i]) for ja in top_a)
            Fv += w_r * sum(ci[jr] * (X[jr] - X[i]) for jr in top_r)
            Fv /= 6.0
            X_new = np.clip(X[i] + ALPHA * Fv, 0, 1)
            nn = np.argmin(np.linalg.norm(Xn - X_new, axis=1))
            X[i] = Xn[nn]; visit[nn] += 1.0
            bja = top_a[np.argmax(ci[top_a])]
            W[i] = (1-BETA)*W[i] + BETA*W[bja]; W[i] /= W[i].sum()

    probs = visit + 1.0; probs /= probs.sum()
    return rng2.choice(N_a, size=n_target, replace=True, p=probs)

print('AISO 오버샘플러 준비 완료')

AISO 오버샘플러 준비 완료


In [19]:
# ── 전체 12개 메서드 실행 ─────────────────────────────────────
results = {}
X_norm   = X_imbal[y_imbal==0]   # 13,449 normal
X_anom   = X_imbal[y_imbal==1]   # 1,345  anomaly (seen)
N_TARGET = N_NORMAL               # 1:1 균형

def run(name, X_tr, y_tr):
    results[name] = evaluate(X_tr, y_tr, name)

# ── 카테고리 1: 베이스라인 ─────────────────────────────────────
print('─'*62)
print('카테고리 1: 베이스라인')
run('원본(불균형)', X_imbal, y_imbal)

clf_cw = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
clf_cw.fit(X_imbal, y_imbal,
           sample_weight=np.where(y_imbal==1, N_NORMAL/N_SEEN, 1.0))
prob_cw = clf_cw.predict_proba(X_test)[:,1]
pred_cw = (prob_cw>=0.5).astype(int)
results['Class Weight'] = {
    'PR-AUC': average_precision_score(y_test, prob_cw),
    'F1':     f1_score(y_test, pred_cw, zero_division=0),
    'AUC':    roc_auc_score(y_test, prob_cw),
}
print(f'  {"Class Weight":<22} PR-AUC={results["Class Weight"]["PR-AUC"]:.4f}'
      f'  F1={results["Class Weight"]["F1"]:.4f}  AUC={results["Class Weight"]["AUC"]:.4f}')

# ── 카테고리 2: 룰 기반 오버샘플링 ───────────────────────────
print('─'*62)
print('카테고리 2: 룰 기반 오버샘플링 (이상 공간 탐색)')

print('  Random...', end=' ')
run('Random', *build_train(X_norm, X_anom, run_random_oversample(X_anom, N_TARGET, SEED)))

print('  K-Means...', end=' ')
run('K-Means', *build_train(X_norm, X_anom, run_kmeans_oversample(X_anom, N_TARGET, SEED)))

print('  Greedy...', end=' ')
run('Greedy', *build_train(X_norm, X_anom, run_greedy_oversample(X_anom, N_TARGET, SEED)))

# ── 카테고리 3: 오버샘플링 (합성 생성) ────────────────────────
print('─'*62)
print('카테고리 3: 오버샘플링 (합성 샘플 생성)')

print('  RandomOverSampler...', end=' ')
run('RandomOver', *RandomOverSampler(random_state=SEED).fit_resample(X_imbal, y_imbal))

print('  SMOTE...', end=' ')
run('SMOTE', *SMOTE(random_state=SEED, k_neighbors=5).fit_resample(X_imbal, y_imbal))

print('  ADASYN...', end=' ')
try:
    run('ADASYN', *ADASYN(random_state=SEED, n_neighbors=5).fit_resample(X_imbal, y_imbal))
except Exception as e:
    results['ADASYN'] = results['SMOTE'].copy()
    print(f'failed ({e}) — using SMOTE')

# ── 카테고리 4: 최적화 기반 오버샘플링 ────────────────────────
print('─'*62)
print('카테고리 4: 최적화 기반 오버샘플링 (이상 공간 탐색 → 가중 리샘플)')

print('  ACO...', end=' ')
run('ACO', *build_train(X_norm, X_anom, run_aco_oversample(X_anom, N_TARGET, SEED)))

print('  PSO...', end=' ')
run('PSO', *build_train(X_norm, X_anom, run_pso_oversample(X_anom, N_TARGET, SEED)))

print('  AISO...', end=' ')
run('AISO', *build_train(X_norm, X_anom, run_aiso_oversample(X_anom, N_TARGET, SEED)))

print('─'*62)
print(f'완료! 총 {len(results)}개 메서드')

──────────────────────────────────────────────────────────────
카테고리 1: 베이스라인
  원본(불균형)                PR-AUC=0.9350  F1=0.7177  AUC=0.9199
  Class Weight           PR-AUC=0.9447  F1=0.7449  AUC=0.9355
──────────────────────────────────────────────────────────────
카테고리 2: 룰 기반 오버샘플링 (이상 공간 탐색)
  Random...   Random                 PR-AUC=0.9438  F1=0.7470  AUC=0.9322
  K-Means...   K-Means                PR-AUC=0.9375  F1=0.7272  AUC=0.9255
  Greedy...   Greedy                 PR-AUC=0.9326  F1=0.7417  AUC=0.9252
──────────────────────────────────────────────────────────────
카테고리 3: 오버샘플링 (합성 샘플 생성)
  RandomOverSampler...   RandomOver             PR-AUC=0.9366  F1=0.7467  AUC=0.9222
  SMOTE...   SMOTE                  PR-AUC=0.9375  F1=0.7458  AUC=0.9261
  ADASYN...   ADASYN                 PR-AUC=0.9318  F1=0.7379  AUC=0.9294
──────────────────────────────────────────────────────────────
카테고리 4: 최적화 기반 오버샘플링 (이상 공간 탐색 → 가중 리샘플)
  ACO...   ACO                    PR-AUC=0.9426  F1=0.7518 

In [20]:
# ── 결과 시각화 ───────────────────────────────────────────────
CATEGORIES = {
    '베이스라인'      : ['원본(불균형)', 'Class Weight'],
    '룰 기반'         : ['Random', 'K-Means', 'Greedy'],
    '오버샘플링'      : ['RandomOver', 'SMOTE', 'ADASYN'],
    '최적화 샘플링'   : ['ACO', 'PSO', 'AISO'],
}
CAT_COLORS = {
    '베이스라인'   : '#888888',
    '룰 기반'      : '#4C72B0',
    '오버샘플링'   : '#CCB974',
    '최적화 샘플링': '#C44E52',
}
METHOD_COLOR = {m: CAT_COLORS[c] for c, ms in CATEGORIES.items() for m in ms}
METHOD_COLOR['AISO'] = '#8B0000'

ranking = sorted(results, key=lambda k: results[k]['PR-AUC'], reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, metric in zip(axes[:2], ['PR-AUC', 'F1']):
    vals   = [results[m][metric] for m in ranking]
    colors = [METHOD_COLOR.get(m, '#aaa') for m in ranking]
    bars   = ax.barh(ranking[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
    for bar, v in zip(bars, vals[::-1]):
        ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=8.5)
    ax.set_xlabel(metric)
    ax.set_title(f'NSL-KDD — {metric}\n(train 10% anomaly / test: novel attack types)')
    ax.set_xlim(0, max(vals) * 1.2)

ax = axes[2]
for m in ranking:
    x, y2 = results[m]['AUC'], results[m]['PR-AUC']
    ax.scatter(x, y2, s=160, color=METHOD_COLOR.get(m,'#aaa'), zorder=5)
    ax.annotate(m, (x, y2), xytext=(4, 4), textcoords='offset points', fontsize=8)
ax.set_xlabel('AUC-ROC'); ax.set_ylabel('PR-AUC')
ax.set_title('AUC-ROC vs PR-AUC')

from matplotlib.patches import Patch
leg = [Patch(facecolor=CAT_COLORS[c], label=c) for c in CAT_COLORS]
axes[0].legend(handles=leg, fontsize=9, loc='lower right')

plt.suptitle('NSL-KDD Showdown — 네트워크 침입 탐지 (12 methods)\n'
             '학습: 41D, 10% anomaly  |  평가: KDDTest+ (미지 공격 포함)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('aiso_showdown_nslkdd.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n' + '='*70)
print(f'  {"전략":<18} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}  카테고리')
print('-'*70)
for rank, m in enumerate(ranking, 1):
    cat  = next((c for c, ms in CATEGORIES.items() if m in ms), '?')
    star = ' ★' if m == 'AISO' else ''
    print(f'  {rank:>2}위 {m:<16} {results[m]["PR-AUC"]:>8.4f} '
          f'{results[m]["F1"]:>8.4f} {results[m]["AUC"]:>8.4f}  {cat}{star}')
print('='*70)


  전략                   PR-AUC       F1      AUC  카테고리
----------------------------------------------------------------------
   1위 AISO               0.9480   0.7318   0.9338  최적화 샘플링 ★
   2위 Class Weight       0.9447   0.7449   0.9355  베이스라인
   3위 Random             0.9438   0.7470   0.9322  룰 기반
   4위 ACO                0.9426   0.7518   0.9302  최적화 샘플링
   5위 K-Means            0.9375   0.7272   0.9255  룰 기반
   6위 SMOTE              0.9375   0.7458   0.9261  오버샘플링
   7위 RandomOver         0.9366   0.7467   0.9222  오버샘플링
   8위 원본(불균형)            0.9350   0.7177   0.9199  베이스라인
   9위 Greedy             0.9326   0.7417   0.9252  룰 기반
  10위 ADASYN             0.9318   0.7379   0.9294  오버샘플링
  11위 PSO                0.9314   0.7334   0.9096  최적화 샘플링
